<a href="https://colab.research.google.com/github/ebi19912/AI/blob/main/VIT%2BConvNeXtBase%2BVGG16_Chest_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
khanfashee_nih_chest_x_ray_14_224x224_resized_path = kagglehub.dataset_download('khanfashee/nih-chest-x-ray-14-224x224-resized')

print('Data source import complete.')


In [ ]:
!pip install torch torchvision
import torch
import torchvision
# =========================
# Standard Library Imports
# =========================
import os
import math
import time
import random

# =========================
# Core Scientific Stack
# =========================
import numpy as np               # Numerical computing
import pandas as pd              # Data manipulation & analysis

# =========================
# Visualization
# =========================
import matplotlib.pyplot as plt  # Plotting
from matplotlib import rcParams  # Matplotlib configuration
import seaborn as sns            # Statistical data visualization

# =========================
# Computer Vision / Imaging
# =========================
import cv2                       # OpenCV for image processing
from PIL import Image            # PIL (Pillow) for image I/O

# =========================
# Scikit-learn (Preprocessing & Metrics)
# =========================
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

# =========================
# PyTorch Ecosystem
# =========================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader    # Custom datasets & batching
from torchvision import transforms                  # Common image transforms
from torch.cuda.amp import autocast, GradScaler     # Mixed precision training (AMP)
import itertools


In [ ]:
dataframe = pd.read_csv("/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv")

# Create Patient_ID column
dataframe['Patient_ID'] = dataframe['Image Index'].apply(lambda x: x.split('_')[0])

# Get unique patients and shuffle them
unique_patients = dataframe['Patient_ID'].unique()
np.random.shuffle(unique_patients)

# Define split sizes
train_size = int(0.7 * len(unique_patients))
val_size = int(0.15 * len(unique_patients))

# Split patients
train_patients = unique_patients[:train_size]
val_patients = unique_patients[train_size : train_size + val_size]
test_patients = unique_patients[train_size + val_size:]

# Create dataframes based on patient split
trainset = dataframe[dataframe['Patient_ID'].isin(train_patients)].reset_index(drop=True)
valset = dataframe[dataframe['Patient_ID'].isin(val_patients)].reset_index(drop=True)
testset = dataframe[dataframe['Patient_ID'].isin(test_patients)].reset_index(drop=True)

print("Train set size:", len(trainset))
print("Validation set size:", len(valset))
print("Test set size:", len(testset))

# Enumerating all column names based on the full dataframe
columns = ["Image"]
for i in dataframe["Finding Labels"].values:
    for j in i.split("|"):
        if j not in columns:
            columns.append(j)
labels = columns.copy()
labels.remove("Image")

# Now, create the one-hot encoded columns for the split dataframes
for df in [trainset, valset, testset]:
    for label in labels:
        df[label] = df["Finding Labels"].apply(lambda x: 1 if label in x else 0)

# Plotting first 16 images with their disease labels
img_dir = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
plt.figure(figsize = (15,15))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(plt.imread(os.path.join(img_dir, trainset["Image Index"][i])), cmap = "gray")
    plt.title(trainset["Finding Labels"][i]) # Use Finding Labels from the split dataframe
plt.tight_layout()
plt.show()

In [ ]:
def isOverlap(s1, s2):
    total = set(s1).intersection(set(s2))
    return [len(total), total]

def overlapcheck(trainset, valset, testset):
    patid_train = []
    patid_val = []
    patid_test = []
    # Corrected column name from 'Image' to 'Image Index'
    for name in trainset['Image Index'].values:
        patid_train.append(int(name.split("_")[0]))

    # Corrected column name from 'Image' to 'Image Index'
    for name in valset['Image Index'].values:
        patid_val.append(int(name.split("_")[0]))

    # Corrected column name from 'Image' to 'Image Index'
    for name in testset['Image Index'].values:
        patid_test.append(int(name.split("_")[0]))
    trte = isOverlap(patid_train, patid_test)
    teva = isOverlap(patid_test, patid_val)
    trva = isOverlap(patid_train, patid_val)
    print("Patient Overlap - Train and Test: ", trte[0])
    print("Patient Overlap - Test and Validation: ", teva[0])
    print("Patient Overlap - Train and Validation: ", trva[0])
    return trte, teva, trva

#Checking for overlaps between trainset, testset and validation set
trte, teva, trva = overlapcheck(trainset, valset, testset)

#Removing overlapping patients
# Corrected column name from 'Image' to 'Image Index' in the loop condition
for i in trva[1]:
    for name in trainset['Image Index'].values:
        if(int(name.split("_")[0]) == i):
            # Corrected column name from 'Image' to 'Image Index' for dropping rows
            trainset.drop(trainset.loc[trainset['Image Index'] == name].index, inplace=True)

#Checking for overlaps after removing common patients
trte, teva, trva = overlapcheck(trainset, valset, testset)

In [ ]:
def label_counts(df):
    label_counts = df[labels].sum().sort_values(ascending=False)
    return label_counts

print("Training Data Label Counts:")
print(label_counts(trainset))
print("\nValidation Data Label Counts:")
print(label_counts(valset))
print("\nTest Data Label Counts:")
print(label_counts(testset))

In [ ]:
# Define the desired labels
desired_labels = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax",
    "No Finding"]
# Function to filter the DataFrame
def filter_dataframe(df, desired_labels):
    # Create a boolean mask for rows with any of the desired labels
    mask = df[desired_labels].any(axis=1)

    # Filter the DataFrame based on the mask
    filtered_df = df[mask].copy() # Add .copy() to avoid SettingWithCopyWarning

    # Select only desired columns
    # Corrected column name from "Image" to "Image Index"
    filtered_df = filtered_df[["Image Index"] + desired_labels]

    return filtered_df

# Apply the filter to each DataFrame
trainset_filtered = filter_dataframe(trainset, desired_labels)
valset_filtered = filter_dataframe(valset, desired_labels)
testset_filtered = filter_dataframe(testset, desired_labels)

In [ ]:

print("Training Set Filtered:")
print(trainset_filtered)
print("\nValidation Set Filtered:")
print(valset_filtered)
print("\nTest Set Filtered:")
testset_filtered


In [ ]:
num = np.random.randint(trainset_filtered.shape[0])
# Corrected column name from 'Image' to 'Image Index'
sample = plt.imread(os.path.join(img_dir,trainset_filtered.iloc[[num]]["Image Index"].values[0]))
plt.figure(figsize=(15, 15))
# Corrected column name from 'Image' to 'Image Index'
plt.title(dataframe[dataframe["Image Index"] == trainset_filtered.iloc[[num]]["Image Index"].values[0]].values[0][1])
plt.imshow(sample, cmap = 'gray')
plt.colorbar()
trainset_filtered.iloc[[num]]

print("Maximum Pixel Value: ", sample.max())
print("Minimum Pixel Value: ", sample.min())
print(f"Image dimension: {sample.shape[0]} x {sample.shape[1]} ")

fig, ax = plt.subplots(figsize=(25, 10))
plt.xlabel("Pixel Values")
print("Mean - Pixel Value: ", sample.mean())
print("Std Deviation Pixel Value: ", sample.std())
sns.histplot(sample.ravel(), ax = ax, kde = True)

In [ ]:
#Correct the labels variable
labels = desired_labels
labels

In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_dir, labels, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.labels = labels  # list of label column names

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Corrected column name from "Image" to "Image Index"
        img_name = os.path.join(self.image_dir, self.df.loc[idx, "Image Index"])
        image = Image.open(img_name).convert('RGB')
        label = torch.tensor(self.df.loc[idx, self.labels].values.astype(np.float32), dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

class PerImageStandardize(object):
    def __call__(self, tensor):
        # tensor: CxHxW
        mean = tensor.mean()
        std = tensor.std()
        std = std if std > 0 else torch.tensor(1.0, device=tensor.device, dtype=tensor.dtype)
        return (tensor - mean) / std

def make_train_transform():
    return transforms.Compose([
        # Zoom ±20%, rotation ±20°, translation ±20%
        transforms.RandomAffine(
            degrees=20,
            translate=(0.2, 0.2),
            scale=(0.8, 1.2)  #  zoom_range=0.2
        ),
        transforms.RandomHorizontalFlip(p=1.0),  #  horizontal_flip=True
        # Brightness/Contrast/Saturation
        transforms.ColorJitter(
            brightness=(0.3, 1.2),
            contrast=(0.4, 0.9),
            saturation=(0.4, 0.9)
        ),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        PerImageStandardize(),  # samplewise center/std (مثل ImageDataGenerator برای train)
    ])

def make_valtest_transform(mean, std):
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)  # featurewise center/std (مثل imagegen برای val/test)
    ])

@torch.no_grad()
def compute_featurewise_stats(dataframe, image_dir, sample_size=1024):

    n = len(dataframe)
    idxs = np.random.choice(n, size=min(sample_size, n), replace=False)

    to_tensor = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    sum_c = torch.zeros(3)
    sumsq_c = torch.zeros(3)
    count = 0

    for i in idxs:
        # Corrected column name from "Image" to "Image Index"
        img_path = os.path.join(image_dir, dataframe.iloc[i]["Image Index"])
        img = Image.open(img_path).convert('RGB')
        t = to_tensor(img)  # CxHxW  [0,1]
        H, W = t.shape[1], t.shape[2]
        num_pixels = H * W

        sum_c += t.view(3, -1).sum(dim=1)
        sumsq_c += (t.view(3, -1) ** 2).sum(dim=1)
        count += num_pixels

    mean = (sum_c / count).tolist()
    var = (sumsq_c / count - (sum_c / count) ** 2).tolist()
    std = torch.sqrt(torch.tensor(var)).tolist()

    return mean, std

img_dir = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
batch = 32

feature_mean, feature_std = compute_featurewise_stats(trainset_filtered, img_dir, sample_size=1024)

train_transform = make_train_transform()
valtest_transform = make_valtest_transform(feature_mean, feature_std)

train_dataset = ChestXrayDataset(trainset_filtered, img_dir, labels, transform=train_transform)
val_dataset   = ChestXrayDataset(valset_filtered,   img_dir, labels, transform=valtest_transform)
test_dataset  = ChestXrayDataset(testset_filtered,  img_dir, labels, transform=valtest_transform)

train_loader = DataLoader(train_dataset, batch_size=batch, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch, shuffle=False, num_workers=2, pin_memory=True)

print("Featurewise mean (RGB):", feature_mean)
print("Featurewise std  (RGB):", feature_std)

raw_only = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

rand_idx = random.randint(0, len(train_dataset) - 1)
# Corrected column name from "Image" to "Image Index"
img_name = os.path.join(img_dir, trainset_filtered.iloc[rand_idx]["Image Index"])
pil_img = Image.open(img_name).convert('RGB')

sample_original = raw_only(pil_img)
sample_standard = PerImageStandardize()(raw_only(pil_img))

print("Mean of Pixel Values - Standardized: ", sample_standard.mean().item())
print("Std  of Pixel Values - Standardized: ", sample_standard.std().item())
print("Mean of Pixel Values - Original: ", sample_original.mean().item())
print("Std  of Pixel Values - Original: ", sample_original.std().item())

plt.figure(figsize=(6,6))
std_img_gray = sample_standard.mean(dim=0).clamp(-3, 3)
plt.imshow(std_img_gray.cpu().numpy(), cmap='gray')
plt.title("Standardized Sample (mean over RGB)")
plt.colorbar()
plt.show()

plt.figure(figsize=(12,5))
plt.hist(sample_standard.flatten().cpu().numpy(), bins=60, alpha=0.7, label='Standardized')
plt.hist(sample_original.flatten().cpu().numpy(), bins=60, alpha=0.7, label='Original')
plt.xlabel("Pixel Values")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
# Calculate positive and negative frequencies for each label
positive_freqs = trainset_filtered[labels].sum().values / trainset_filtered.shape[0]  # Assuming labels are 1 for positive, 0 for negative
negative_freqs = 1 - positive_freqs

data = {
    'Class': labels,
    'Positive': positive_freqs, #* negative_freqs, #Removed
    'Negative': negative_freqs #* positive_freqs #Removed
}

X_axis = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(X_axis-0.2, data['Positive'], width=0.4, color='b', label = "Positive")
ax.bar(X_axis+0.2, data['Negative'], width=0.4, color='r', label = 'Negative')
plt.xticks(X_axis, labels, rotation = 90)
plt.legend()
plt.figure(figsize=(20,15))

In [ ]:
save_dir = '/kaggle/working/'

In [ ]:
class GAP(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
    def forward(self, x):
        return self.pool(x).flatten(1)

def freeze_all(module: nn.Module):
    for p in module.parameters():
        p.requires_grad = False

def unfreeze_last_k_params(module: nn.Module, k: int):
    params = [p for p in module.parameters()]
    for p in params[-k:]:
        p.requires_grad = True

class MultiBackboneChest(nn.Module):
    def __init__(self, num_labels: int, unfreeze_last=5):
        super().__init__()
        self.num_labels = num_labels

        # ---------- VGG16 (feature extractor) ----------
        vgg_weights = torchvision.models.VGG16_Weights.IMAGENET1K_V1
        vgg = torchvision.models.vgg16(weights=vgg_weights)
        self.vgg_features = vgg.features
        self.vgg_gap = GAP()
        self.vgg_fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True)
        )
        freeze_all(self.vgg_features)
        unfreeze_last_k_params(self.vgg_features, unfreeze_last)

        # ---------- ConvNeXt-Base ----------
        cnx = torchvision.models.convnext_base(weights=torchvision.models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
        self.cnx_features = cnx.features
        self.cnx_norm = cnx.classifier[0] if isinstance(cnx.classifier[0], nn.LayerNorm) else nn.Identity()
        self.cnx_gap = GAP()
        self.cnx_fc = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(inplace=True)
        )
        freeze_all(self.cnx_features)
        unfreeze_last_k_params(self.cnx_features, unfreeze_last)

        # ---------- ViT-base-patch16-224 ----------
        from transformers import ViTModel, AutoImageProcessor
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224')
        freeze_all(self.vit)
        unfreeze_last_k_params(self.vit, unfreeze_last)

        processor = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
        self.vit_mean = torch.tensor(processor.image_mean).view(1, 3, 1, 1)
        self.vit_std  = torch.tensor(processor.image_std).view(1, 3, 1, 1)


        self.vit_fc = nn.Sequential(
            nn.Linear(self.vit.config.hidden_size, 256),
            nn.ReLU(inplace=True)
        )

        # ---------- Final head ----------
        self.classifier = nn.Linear(256 * 3, num_labels)
        imagenet_transforms = torchvision.models.VGG16_Weights.IMAGENET1K_V1.transforms()
        self.imagenet_mean = torch.tensor(imagenet_transforms.mean).view(1, 3, 1, 1)
        self.imagenet_std  = torch.tensor(imagenet_transforms.std).view(1, 3, 1, 1)

    def forward(self, x):
        """
        x: Bx3x224x224   [0,1]
        """
        device = x.device
        imagenet_mean = self.imagenet_mean.to(device)
        imagenet_std  = self.imagenet_std.to(device)
        vit_mean      = self.vit_mean.to(device)
        vit_std       = self.vit_std.to(device)

        x_torch = (x - imagenet_mean) / imagenet_std
        x_vit   = (x - vit_mean) / vit_std

        # ----- VGG16 -----
        vgg_feat = self.vgg_features(x_torch)          # [B, 512, H', W']
        vgg_vec  = self.vgg_gap(vgg_feat)              # [B, 512]
        vgg_vec  = self.vgg_fc(vgg_vec)                # [B, 256]

        # ----- ConvNeXt -----
        cnx_feat = self.cnx_features(x_torch)          # [B, 1024, H', W']
        if isinstance(self.cnx_norm, nn.LayerNorm):
            cnx_feat = self.cnx_norm(cnx_feat)
        cnx_vec  = self.cnx_gap(cnx_feat)              # [B, 1024]
        cnx_vec  = self.cnx_fc(cnx_vec)                # [B, 256]

        # ----- ViT -----
        vit_out = self.vit(pixel_values=x_vit)         # last_hidden_state: [B, N, D]
        cls_tok = vit_out.last_hidden_state[:, 0, :]   # [B, D]
        vit_vec = self.vit_fc(cls_tok)                 # [B, 256]

        merged = torch.cat([cnx_vec, vgg_vec, vit_vec], dim=1)  # [B, 768]
        logits = self.classifier(merged)                         # [B, num_labels]
        return logits

In [ ]:
num_labels = len(labels)
model = MultiBackboneChest(num_labels=num_labels, unfreeze_last=5)

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
pos_counts = torch.tensor([ (trainset_filtered[l].sum()) for l in labels ], dtype=torch.float32)
neg_counts = torch.tensor([ (len(trainset_filtered) - trainset_filtered[l].sum()) for l in labels ], dtype=torch.float32)
pos_weight_vals = (neg_counts / pos_counts.clamp_min(1.0)).clamp(1.0, 50.0)

pos_weight = pos_weight_vals.to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

class EarlyStopping:
    def __init__(self, patience=5, mode='max', delta=1e-6, ckpt_path='best_model.pt'):
        self.patience = patience
        self.mode = mode
        self.delta = delta
        self.ckpt_path = ckpt_path
        self.best = -math.inf if mode=='max' else math.inf
        self.counter = 0
        self.stop = False

    def __call__(self, metric, model):
        improved = (metric > self.best + self.delta) if self.mode=='max' else (metric < self.best - self.delta)
        if improved:
            self.best = metric
            self.counter = 0
            torch.save({'state_dict': model.state_dict()}, self.ckpt_path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

early = EarlyStopping(patience=5, mode='max', ckpt_path='best_model.pt')

scaler = GradScaler(enabled=(device.type=='cuda'))
num_epochs = 15
grad_clip_norm = 1.0

def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()
    running_loss = 0.0
    all_logits, all_targets = [], []

    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True).float()

        if train:
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=(device.type=='cuda')):
                logits = model(inputs)               # [B,C] logits
                loss = criterion(logits, targets)    # BCE-with-logits
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if grad_clip_norm is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad(), autocast(enabled=(device.type=='cuda')):
                logits = model(inputs)
                loss = criterion(logits, targets)

        running_loss += loss.item() * inputs.size(0)
        all_logits.append(logits.detach().float().cpu())
        all_targets.append(targets.detach().float().cpu())

    epoch_loss = running_loss / len(loader.dataset)
    all_logits = torch.cat(all_logits, dim=0).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()

    probs = 1 / (1 + np.exp(-all_logits))  # sigmoid
    metrics = compute_metrics_multi_label(all_targets, probs, labels)
    return epoch_loss, metrics, probs, all_targets

def compute_metrics_multi_label(y_true, y_prob, class_names):
    metrics = {}
    try:
        metrics['auroc_macro'] = roc_auc_score(y_true, y_prob, average='macro')
        metrics['auroc_micro'] = roc_auc_score(y_true.ravel(), y_prob.ravel())
    except ValueError:
        metrics['auroc_macro'] = np.nan
        metrics['auroc_micro'] = np.nan

    ap_per_class = []
    for i in range(y_true.shape[1]):
        yi, pi = y_true[:, i], y_prob[:, i]
        if yi.max() > 0 and yi.min() < 1:
            ap = average_precision_score(yi, pi)
            ap_per_class.append(ap)
        else:
            ap_per_class.append(np.nan)
    metrics['mAP_macro'] = np.nanmean(ap_per_class)

    per_class_auroc = {}
    for i, name in enumerate(class_names):
        yi, pi = y_true[:, i], y_prob[:, i]
        try:
            per_class_auroc[name] = roc_auc_score(yi, pi)
        except ValueError:
            per_class_auroc[name] = np.nan
    metrics['auroc_per_class'] = per_class_auroc
    return metrics

best_epoch = -1
for epoch in range(1, num_epochs+1):
    t0 = time.time()
    train_loss, train_metrics, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_metrics, val_probs, val_targets = run_epoch(val_loader, train=False)

    score = val_metrics['auroc_macro']
    scheduler.step(score if not np.isnan(score) else 0.0)
    early(score if not np.isnan(score) else -1e9, model)

    et = time.time() - t0
    print(f"[{epoch:02d}/{num_epochs}] "
          f"TrainLoss={train_loss:.4f} | ValLoss={val_loss:.4f} | "
          f"Val AUROC macro={val_metrics['auroc_macro']:.4f} | "
          f"mAP={val_metrics['mAP_macro']:.4f} | time={et:.1f}s")

    top_show = min(5, len(labels))
    top_items = list(val_metrics['auroc_per_class'].items())[:top_show]
    print("  AUROC per-class (sample):", ", ".join([f"{k}:{v:.3f}" for k,v in top_items]))

    if early.stop:
        print("Early stopping triggered.")
        break

ckpt = torch.load('best_model.pt', map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

In [ ]:
# Evaluation & Visualization (Multi-label)

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

test_logits = []
test_targets_eval = [] # Use a different variable name to avoid conflict

with torch.no_grad(), autocast(enabled=(device.type=='cuda')):
    for inputs, targets in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        logits = model(inputs)
        test_logits.append(logits.cpu().numpy())
        test_targets_eval.append(targets.cpu().numpy())

test_logits = np.concatenate(test_logits, axis=0)
test_targets_eval = np.concatenate(test_targets_eval, axis=0)
test_probs = 1 / (1 + np.exp(-test_logits)) # Apply sigmoid to get probabilities

# Initialize lists to store metrics
precision_list = []
recall_list = []
f1_list = []
accuracy_list = []
mcc_list = []
auc_roc_list = []
auc_pr_list = []
specificity_list = []
best_threshold_list = []

# Iterate through each label to calculate metrics and find best threshold
for i, label in enumerate(labels):
    y_true = test_targets_eval[:, i]
    y_prob = test_probs[:, i]

    # Find the best threshold based on F1-score
    best_f1 = -1
    best_threshold = 0.0
    thresholds_to_try = np.linspace(0.05, 0.95, 50)

    for threshold in thresholds_to_try:
        y_pred_binary = (y_prob >= threshold).astype(int)
        # Handle cases where all predictions are the same
        if np.all(y_pred_binary == 0) or np.all(y_pred_binary == 1):
             current_f1 = 0.0
        else:
             current_f1 = f1_score(y_true, y_pred_binary)

        if current_f1 > best_f1:
            best_f1 = current_f1
            best_threshold = threshold

    best_threshold_list.append(best_threshold)
    y_pred_binary_best = (y_prob >= best_threshold).astype(int)

    # Calculate metrics using the best threshold
    precision = precision_score(y_true, y_pred_binary_best)
    recall = recall_score(y_true, y_pred_binary_best)
    f1 = f1_score(y_true, y_pred_binary_best)
    accuracy = accuracy_score(y_true, y_pred_binary_best)
    mcc = matthews_corrcoef(y_true, y_pred_binary_best)

    # Calculate AUC-ROC and AUC-PR using probabilities
    # Handle cases with only one class present
    if len(np.unique(y_true)) == 1:
        auc_roc = np.nan # AUC is undefined when only one class is present
        auc_pr = np.nan # AUC is undefined when only one class is present
    else:
        auc_roc = roc_auc_score(y_true, y_prob)
        auc_pr = average_precision_score(y_true, y_prob)

    # Calculate confusion matrix and specificity
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary_best).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)
    accuracy_list.append(accuracy)
    mcc_list.append(mcc)
    auc_roc_list.append(auc_roc)
    auc_pr_list.append(auc_pr)
    specificity_list.append(specificity)


# Create a DataFrame to display the metrics
metrics_df = pd.DataFrame({
    'Label': labels,
    'Best Threshold': best_threshold_list,
    'Precision': precision_list,
    'Recall': recall_list,
    'F1-Score': f1_list,
    'Accuracy': accuracy_list,
    'MCC': mcc_list,
    'AUC-ROC': auc_roc_list,
    'AUC-PR': auc_pr_list,
    'Specificity': specificity_list
})

print("Evaluation Metrics per Label:")
display(metrics_df)

# Visualize ROC and Precision-Recall curves
plt.figure(figsize=(15, 10))

for i, label in enumerate(labels):
    y_true = test_targets_eval[:, i]
    y_prob = test_probs[:, i]

    # Plot ROC Curve
    plt.subplot(1, 2, 1)
    if len(np.unique(y_true)) > 1:
        fpr, tpr, thresholds = roc_curve(y_true, y_prob)
        plt.plot(fpr, tpr, label=f'{label} (AUC = {auc_roc_list[i]:.2f})')
        # Find optimal threshold using Youden's J statistic
        youden_j = tpr - fpr
        optimal_threshold_roc = thresholds[np.argmax(youden_j)]
        plt.plot(fpr[np.argmax(youden_j)], tpr[np.argmax(youden_j)], 'o', markersize=5, color='red') # Mark optimal threshold on ROC curve
        plt.text(fpr[np.argmax(youden_j)] + 0.02, tpr[np.argmax(youden_j)] - 0.02, f'{optimal_threshold_roc:.2f}', color='red')


    # Plot Precision-Recall Curve
    plt.subplot(1, 2, 2)
    if len(np.unique(y_true)) > 1:
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        plt.plot(recall, precision, label=f'{label} (AP = {auc_pr_list[i]:.2f})')


plt.subplot(1, 2, 1)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:

print("Calculating metrics...")
metrics_list = []
class_supports = []

for i, label in enumerate(labels):
    y_true = test_targets_eval[:, i]
    y_prob = test_probs[:, i]

    support = np.sum(y_true)
    class_supports.append(support)

    best_f1, best_thresh = -1, 0.0
    for threshold in np.linspace(0.05, 0.95, 50):
        y_pred_binary = (y_prob >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred_binary, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = threshold

    y_pred_best = (y_prob >= best_thresh).astype(int)

    precision = precision_score(y_true, y_pred_best, zero_division=0)
    recall = recall_score(y_true, y_pred_best, zero_division=0)
    f1 = f1_score(y_true, y_pred_best, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_best).ravel()
    auc_roc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    auc_pr = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan

    metrics_list.append({
        'Label': label,
        'Support (Positive Samples)': support,
        'Best Threshold': f"{best_thresh:.3f}",
        'F1-Score': f1,
        'AUC-ROC': auc_roc,
        'AUC-PR': auc_pr,
        'Precision': precision,
        'Recall': recall,
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'Accuracy': accuracy_score(y_true, y_pred_best)
    })

metrics_df = pd.DataFrame(metrics_list)
print("Evaluation Metrics per Label:")
display(metrics_df.round(4))

total_support = np.sum(class_supports)
if total_support > 0:
    weighted_precision = np.average([m['Precision'] for m in metrics_list], weights=class_supports)
    weighted_recall = np.average([m['Recall'] for m in metrics_list], weights=class_supports)
    weighted_f1 = np.average([m['F1-Score'] for m in metrics_list], weights=class_supports)
else:
    weighted_precision, weighted_recall, weighted_f1 = 0, 0, 0


print("\n" + "="*50)
print("Overall Weighted Metrics (based on number of positive samples per class)")
print("="*50)
print(f"Weighted Average Precision: {weighted_precision:.4f}")
print(f"Weighted Average Recall:    {weighted_recall:.4f}")
print(f"Weighted Average F1-Score:  {weighted_f1:.4f}")
print("="*50)



# Visualize ROC and Precision-Recall curves
plt.figure(figsize=(15, 10))

for i, label in enumerate(labels):
    y_true = test_targets_eval[:, i]
    y_prob = test_probs[:, i]

    # Plot ROC Curve
    plt.subplot(1, 2, 1)
    if len(np.unique(y_true)) > 1:
        fpr, tpr, thresholds = roc_curve(y_true, y_prob)
        plt.plot(fpr, tpr, label=f'{label} (AUC = {auc_roc_list[i]:.2f})')
        # Find optimal threshold using Youden's J statistic
        youden_j = tpr - fpr
        optimal_threshold_roc = thresholds[np.argmax(youden_j)]
        plt.plot(fpr[np.argmax(youden_j)], tpr[np.argmax(youden_j)], 'o', markersize=5, color='red') # Mark optimal threshold on ROC curve
        plt.text(fpr[np.argmax(youden_j)] + 0.02, tpr[np.argmax(youden_j)] - 0.02, f'{optimal_threshold_roc:.2f}', color='red')


    # Plot Precision-Recall Curve
    plt.subplot(1, 2, 2)
    if len(np.unique(y_true)) > 1:
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        plt.plot(recall, precision, label=f'{label} (AP = {auc_pr_list[i]:.2f})')


plt.subplot(1, 2, 1)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:

def plot_confusion_matrix(cm, classes, title='Confusion Matrix', cmap=plt.cm.Blues, normalize=False): # Added normalize parameter
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.2f'
    else:
        fmt = 'd'

    plt.imshow(cm, interpolation='nearest', cmap=cmap, vmin=0, vmax=np.max(cm) if not normalize else 1.0) # Added vmin/vmax
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    thresh = cm.max() / 2.
    # Fixed annotation format and logic
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, fmt.format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

# Calculate and plot confusion matrix for each label
plt.figure(figsize=(20, 25))
for i, label in enumerate(labels):
    y_true = test_targets_eval[:, i]
    y_prob = test_probs[:, i]

    # Use the best threshold found previously for this label
    best_threshold = metrics_df[metrics_df['Label'] == label]['Best Threshold'].values[0]
    y_pred_binary_best = (y_prob >= float(best_threshold)).astype(int) # Convert best_threshold to float

    cm = confusion_matrix(y_true, y_pred_binary_best)

    plt.subplot(5, 3, i + 1)
    # Pass normalize=True to normalize per class matrix for better visualization
    plot_confusion_matrix(cm, classes=['Negative', 'Positive'], title=f'Confusion Matrix: {label}', normalize=True)

plt.tight_layout()
plt.show()

# Calculate and plot weighted average confusion matrix
# This is a simplified approach; a true weighted average CM is more complex and dataset dependent
# Here we sum up individual confusion matrices
weighted_cm = np.zeros((2, 2), dtype=int)
for i, label in enumerate(labels):
    y_true = test_targets_eval[:, i]
    y_prob = test_probs[:, i]
     # Use the best threshold found previously for this label
    best_threshold = metrics_df[metrics_df['Label'] == label]['Best Threshold'].values[0]
    y_pred_binary_best = (y_prob >= float(best_threshold)).astype(int) # Convert best_threshold to float
    cm = confusion_matrix(y_true, y_pred_binary_best)
    weighted_cm += cm # Summing up TN, FP, FN, TP across all labels

plt.figure(figsize=(6, 6))
# Do not normalize the weighted average confusion matrix to see the total counts
plot_confusion_matrix(weighted_cm, classes=['Negative', 'Positive'], title='Weighted Average Confusion Matrix', normalize=False)
plt.show()

In [ ]:
def calculate_macro_metrics(y_true, y_prob, labels):
    """Calculates macro-averaged AUC-ROC and mAP."""
    auroc_per_class = []
    ap_per_class = []

    for i in range(y_true.shape[1]):
        yi, pi = y_true[:, i], y_prob[:, i]
        # Only calculate metrics if there is more than one class present in the sample
        if len(np.unique(yi)) > 1:
            try:
                auroc_per_class.append(roc_auc_score(yi, pi))
            except ValueError:
                auroc_per_class.append(np.nan)

            try:
                ap_per_class.append(average_precision_score(yi, pi))
            except ValueError:
                ap_per_class.append(np.nan)
        else:
            auroc_per_class.append(np.nan)
            ap_per_class.append(np.nan)


    macro_auroc = np.nanmean(auroc_per_class)
    macro_mAP = np.nanmean(ap_per_class)

    return macro_auroc, macro_mAP

# Bootstrapping parameters
n_bootstraps = 1000
bootstrap_aurocs = []
bootstrap_maps = []
n_test_samples = len(test_targets_eval)

# Implement bootstrapping
for _ in range(n_bootstraps):
    # Sample indices with replacement
    indices = np.random.choice(n_test_samples, size=n_test_samples, replace=True)

    # Select bootstrapped samples
    y_true_bootstrap = test_targets_eval[indices]
    y_prob_bootstrap = test_probs[indices]

    # Calculate metrics for the bootstrapped sample
    macro_auroc_bootstrap, macro_mAP_bootstrap = calculate_macro_metrics(y_true_bootstrap, y_prob_bootstrap, labels)

    bootstrap_aurocs.append(macro_auroc_bootstrap)
    bootstrap_maps.append(macro_mAP_bootstrap)

# Remove NaN values before calculating percentiles
bootstrap_aurocs = np.array(bootstrap_aurocs)
bootstrap_maps = np.array(bootstrap_maps)

bootstrap_aurocs = bootstrap_aurocs[~np.isnan(bootstrap_aurocs)]
bootstrap_maps = bootstrap_maps[~np.isnan(bootstrap_maps)]

# Calculate 95% confidence intervals
if len(bootstrap_aurocs) > 0:
    ci_auroc_lower = np.percentile(bootstrap_aurocs, 2.5)
    ci_auroc_upper = np.percentile(bootstrap_aurocs, 97.5)
    print(f"95% Confidence Interval for Macro-averaged AUC-ROC: ({ci_auroc_lower:.4f}, {ci_auroc_upper:.4f})")
else:
    print("Could not calculate 95% Confidence Interval for Macro-averaged AUC-ROC due to insufficient valid bootstrap samples.")

if len(bootstrap_maps) > 0:
    ci_mAP_lower = np.percentile(bootstrap_maps, 2.5)
    ci_mAP_upper = np.percentile(bootstrap_maps, 97.5)
    print(f"95% Confidence Interval for mAP: ({ci_mAP_lower:.4f}, {ci_mAP_upper:.4f})")
else:
    print("Could not calculate 95% Confidence Interval for mAP due to insufficient valid bootstrap samples.")


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Calculate macro averages
macro_precision = precision_score(test_targets_eval, (test_probs >= best_threshold_list).astype(int), average='macro', zero_division=0)
macro_recall = recall_score(test_targets_eval, (test_probs >= best_threshold_list).astype(int), average='macro', zero_division=0)
macro_f1 = f1_score(test_targets_eval, (test_probs >= best_threshold_list).astype(int), average='macro', zero_division=0)

# Calculate micro averages
micro_precision = precision_score(test_targets_eval, (test_probs >= best_threshold_list).astype(int), average='micro', zero_division=0)
micro_recall = recall_score(test_targets_eval, (test_probs >= best_threshold_list).astype(int), average='micro', zero_division=0)
micro_f1 = f1_score(test_targets_eval, (test_probs >= best_threshold_list).astype(int), average='micro', zero_division=0)

# Print the results
print("\n" + "="*50)
print("Macro and Micro Averaged Metrics")
print("="*50)
print(f"Macro Average Precision: {macro_precision:.4f}")
print(f"Macro Average Recall:    {macro_recall:.4f}")
print(f"Macro Average F1-Score:  {macro_f1:.4f}")
print("-" * 50)
print(f"Micro Average Precision: {micro_precision:.4f}")
print(f"Micro Average Recall:    {micro_recall:.4f}")
print(f"Micro Average F1-Score:  {micro_f1:.4f}")
print("="*50)

In [ ]:
# Calculate class counts for each dataset split
train_counts = trainset_filtered[labels].sum()
val_counts = valset_filtered[labels].sum()
test_counts = testset_filtered[labels].sum()

# Create a DataFrame to store the statistics
dataset_stats_df = pd.DataFrame({
    'Label': labels,
    'Train Count': train_counts.values,
    'Validation Count': val_counts.values,
    'Test Count': test_counts.values
})

# Display the DataFrame
print("Dataset Statistics per Label:")
display(dataset_stats_df)

In [ ]:
print(model)

In [ ]:
from sklearn.calibration import calibration_curve

plt.figure(figsize=(15, 10))
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")

for i, label in enumerate(labels):
    y_true = test_targets_eval[:, i]
    y_prob = test_probs[:, i]

    # Only plot if there are both positive and negative samples
    if len(np.unique(y_true)) > 1:
        fraction_of_positives, mean_predicted_value = calibration_curve(y_true, y_prob, n_bins=10)
        plt.plot(mean_predicted_value, fraction_of_positives, "s-", label=f"{label} ({auc_pr_list[i]:.2f} AP)") # Using AP from previous metrics

plt.xlabel("Mean predicted value")
plt.ylabel("Fraction of positives")
plt.title("Calibration curves")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()